# NewsQA preprocessing and token alignment

Download the NewsQA subset, convert it into the same SQuAD v1.1 JSON schema used by `train_qa.ipynb` (train-v1.1.json / dev-v1.1.json), and verify character-level and token-level answer alignment before handing it off.

This notebook does not train a model. It produces `train-newsqa.json` and `dev-newsqa.json` in the shared `qa/` folder on Drive, in the exact format `evaluate-v2.0.py` and the main training pipeline already expect.

NewsQA's official release requires separately downloading CNN/DailyMail source articles and running a combine script, due to licensing. To avoid that, this notebook pulls NewsQA from the MRQA 2019 Shared Task, which hosts it pre-combined in a SQuAD-like format.

Note: the download URLs below are the standard MRQA 2019 release URLs, but if they've moved, the download cell will tell you to check the MRQA-Shared-Task-2019 GitHub repo for the current link.

In [1]:
!pip install -q -U transformers
import transformers
print("transformers version:", transformers.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 68.8 MB/s eta 0:00:00
transformers version: 5.20.2


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import json
import gzip
import random
import urllib.request

from transformers import AutoTokenizer

MODEL_CHECKPOINT = "bert-base-uncased"   # match train_qa.ipynb, used only for the alignment sanity checks below
MAX_LENGTH = 384
DOC_STRIDE = 128

PROJECT_DIR = "/content/drive/MyDrive/CS4248 Group Project"
DATA_DIR = os.path.join(PROJECT_DIR, "qa")
NEWSQA_DIR = os.path.join(DATA_DIR, "newsqa_raw")
os.makedirs(NEWSQA_DIR, exist_ok=True)

MRQA_TRAIN_URL = "https://s3.us-east-2.amazonaws.com/mrqa/release/v2/train/NewsQA.jsonl.gz"
MRQA_DEV_URL = "https://s3.us-east-2.amazonaws.com/mrqa/release/v2/dev/NewsQA.jsonl.gz"

TRAIN_RAW_GZ = os.path.join(NEWSQA_DIR, "NewsQA_train.jsonl.gz")
DEV_RAW_GZ = os.path.join(NEWSQA_DIR, "NewsQA_dev.jsonl.gz")

# Proposal calls this a "subset", so cap it rather than keeping the full
# NewsQA train split. Set to None to keep everything.
MAX_NEWSQA_TRAIN_QUESTIONS = 10000
MAX_NEWSQA_DEV_QUESTIONS = 1000
RANDOM_SEED = 42

OUT_TRAIN_JSON = os.path.join(DATA_DIR, "train-newsqa.json")
OUT_DEV_JSON = os.path.join(DATA_DIR, "dev-newsqa.json")

In [4]:
def download(url, dest):
    if os.path.exists(dest):
        print(f"Already downloaded: {dest}")
        return
    print(f"Downloading {url} ...")
    try:
        urllib.request.urlretrieve(url, dest)
        print(f"Saved to {dest}")
    except Exception as e:
        raise RuntimeError(
            f"Could not download {url} ({e}). "
            "If this URL has moved, get the current NewsQA download link from "
            "the MRQA 2019 Shared Task GitHub repo (mrqa/MRQA-Shared-Task-2019), "
            "under its data-download script, and set MRQA_TRAIN_URL / MRQA_DEV_URL "
            "in the cell above, then rerun this cell."
        )

download(MRQA_TRAIN_URL, TRAIN_RAW_GZ)
download(MRQA_DEV_URL, DEV_RAW_GZ)

Saved to /content/drive/MyDrive/CS4248 Group Project/qa/newsqa_raw/NewsQA_train.jsonl.gz
Saved to /content/drive/MyDrive/CS4248 Group Project/qa/newsqa_raw/NewsQA_dev.jsonl.gz


In [5]:
def load_mrqa_jsonl(path):
    documents = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            if "header" in row and "context" not in row:
                continue  # first line is dataset metadata, not a document
            documents.append(row)
    return documents

print("Parsing raw NewsQA files...")
train_docs = load_mrqa_jsonl(TRAIN_RAW_GZ)
dev_docs = load_mrqa_jsonl(DEV_RAW_GZ)

print(f"Raw documents: {len(train_docs)} train, {len(dev_docs)} dev")
print(f"Raw questions: {sum(len(d['qas']) for d in train_docs)} train, "
      f"{sum(len(d['qas']) for d in dev_docs)} dev")

Parsing raw NewsQA files...
Raw documents: 11428 train, 638 dev
Raw questions: 74160 train, 4212 dev


In [6]:
def mrqa_to_squad_format(documents, max_questions=None, seed=RANDOM_SEED):
    """Converts MRQA-format NewsQA documents into the nested SQuAD v1.1
    JSON schema used by train-v1.1.json / dev-v1.1.json, so the result is a
    drop-in match for the existing load_squad_json() loader and for
    evaluate-v2.0.py.
    """
    paragraphs = []
    n_dropped = 0

    for doc in documents:
        context = doc["context"]
        qas_out = []
        for qa in doc["qas"]:
            detected = qa.get("detected_answers", [])
            if not detected:
                n_dropped += 1
                continue

            answers = []
            for det in detected:
                text = det["text"]
                char_start, char_end = det["char_spans"][0]
                # MRQA char_spans can be end-inclusive or end-exclusive
                # depending on release version, so pick whichever slice
                # actually matches the given answer text.
                if context[char_start:char_end + 1] == text:
                    start = char_start
                elif context[char_start:char_end] == text:
                    start = char_start
                else:
                    start = char_start  # kept as-is, flagged by the alignment check below
                answers.append({"text": text, "answer_start": start})

            qid = qa.get("qid", qa.get("id"))
            qas_out.append({
                "id": qid,
                "question": qa["question"],
                "answers": answers,
            })

        if qas_out:
            paragraphs.append({"context": context, "qas": qas_out})

    rng = random.Random(seed)
    rng.shuffle(paragraphs)

    if max_questions is not None:
        selected, total = [], 0
        for p in paragraphs:
            if total >= max_questions:
                break
            selected.append(p)
            total += len(p["qas"])
        paragraphs = selected

    n_questions = sum(len(p["qas"]) for p in paragraphs)
    print(f"Converted {n_questions} questions across {len(paragraphs)} paragraphs "
          f"({n_dropped} dropped, no detected answer)")
    return {"data": [{"title": "NewsQA", "paragraphs": paragraphs}]}

print("Converting to SQuAD v1.1 format...")
train_squad = mrqa_to_squad_format(train_docs, max_questions=MAX_NEWSQA_TRAIN_QUESTIONS)
dev_squad = mrqa_to_squad_format(dev_docs, max_questions=MAX_NEWSQA_DEV_QUESTIONS)

Converting to SQuAD v1.1 format...
Converted 10003 questions across 1544 paragraphs (0 dropped, no detected answer)
Converted 1003 questions across 153 paragraphs (0 dropped, no detected answer)


## Alignment sanity checks

Two checks, matching Member 1's "token alignment" responsibility:

1. **Character-level**: does `context[answer_start : answer_start + len(text)]` actually equal the stored answer text?
2. **Token-level**: using the exact same sliding-window labeling logic as `prepare_train_features` in `train_qa.ipynb`, does decoding the resulting start/end token positions back through the offset mapping recover the gold answer?

If either check comes back with a low match rate, flag it to the team before Member 2 trains on this data, since misaligned labels would silently corrupt training.

In [7]:
def flatten(squad_data):
    return [
        (p["context"], qa)
        for article in squad_data["data"]
        for p in article["paragraphs"]
        for qa in p["qas"]
    ]

def check_char_alignment(squad_data, n=200, seed=RANDOM_SEED):
    rng = random.Random(seed)
    flat = flatten(squad_data)
    sample = rng.sample(flat, min(n, len(flat)))
    mismatches = 0
    examples_shown = 0
    for context, qa in sample:
        ans = qa["answers"][0]
        start = ans["answer_start"]
        extracted = context[start:start + len(ans["text"])]
        if extracted != ans["text"]:
            mismatches += 1
            if examples_shown < 3:
                print(f"  mismatch -> expected {ans['text']!r}, got {extracted!r}")
                examples_shown += 1
    checked = len(sample)
    print(f"Character-level check: {checked - mismatches}/{checked} correct")
    return mismatches

print("Train set:")
check_char_alignment(train_squad)
print("Dev set:")
check_char_alignment(dev_squad)

Train set:
  mismatch -> expected "L'AQUILA, Italy", got "L'Aquila, Italy"
Character-level check: 199/200 correct
Dev set:
Character-level check: 200/200 correct


0

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
assert tokenizer.is_fast, "Need a fast tokenizer for offset mapping"

def check_token_alignment(squad_data, n=200, seed=RANDOM_SEED):
    rng = random.Random(seed)
    flat = flatten(squad_data)
    sample = rng.sample(flat, min(n, len(flat)))
    correct = 0
    not_found = 0

    for context, qa in sample:
        question = qa["question"].lstrip()
        ans = qa["answers"][0]
        start_char = ans["answer_start"]
        end_char = start_char + len(ans["text"])

        tokenized = tokenizer(
            question, context,
            max_length=MAX_LENGTH, truncation="only_second",
            stride=DOC_STRIDE, return_overflowing_tokens=True,
            return_offsets_mapping=True, padding="max_length",
        )

        found = False
        for i, offsets in enumerate(tokenized["offset_mapping"]):
            sequence_ids = tokenized.sequence_ids(i)
            idx = 0
            while sequence_ids[idx] != 1:
                idx += 1
            context_start = idx
            while idx < len(sequence_ids) and sequence_ids[idx] == 1:
                idx += 1
            context_end = idx - 1

            if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
                continue  # answer not inside this sliding window

            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_pos = idx - 1
            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_pos = idx + 1

            decoded = context[offsets[start_pos][0]:offsets[end_pos][1]]
            if decoded.strip() == ans["text"].strip():
                correct += 1
            found = True
            break

        if not found:
            not_found += 1

    checked = len(sample)
    print(f"Token-level check: {correct}/{checked} spans decode back to the gold answer "
          f"({not_found} examples had no window containing the answer)")

print("Train set:")
check_token_alignment(train_squad)
print("Dev set:")
check_token_alignment(dev_squad)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train set:
Token-level check: 199/200 spans decode back to the gold answer (0 examples had no window containing the answer)
Dev set:
Token-level check: 200/200 spans decode back to the gold answer (0 examples had no window containing the answer)


In [9]:
with open(OUT_TRAIN_JSON, "w") as f:
    json.dump(train_squad, f)
with open(OUT_DEV_JSON, "w") as f:
    json.dump(dev_squad, f)

print(f"Saved {OUT_TRAIN_JSON}")
print(f"Saved {OUT_DEV_JSON}")

Saved /content/drive/MyDrive/CS4248 Group Project/qa/train-newsqa.json
Saved /content/drive/MyDrive/CS4248 Group Project/qa/dev-newsqa.json
